# Preparação de dados: série temporal de tráfego do PTT/IX

Este notebook realiza a etapa de preparação da base utilizada nas análises posteriores. A partir do arquivo XML bruto do PTT/IX, são reconstruídas as observações temporais, calculada a completude anual e gerado o conjunto de dados filtrado em Gbps.

Neste notebook:

- lemos o arquivo XML bruto do PTT/IX;
- reconstruímos a série temporal a partir dos timestamps e do intervalo de coleta;
- calculamos o tráfego total como `entrada + saída`;
- avaliamos a completude anual da série;
- mantemos apenas os anos com completude superior a 95%;
- convertemos os valores de bps para Gbps;
- salvamos os principais arquivos processados em `data/processed`.


## 1. Importação de bibliotecas

In [6]:
import sys

print(sys.executable)
print(sys.version)

/Library/Developer/CommandLineTools/usr/bin/python3
3.9.6 (default, Dec  2 2025, 07:27:58) 
[Clang 17.0.0 (clang-1700.6.3.2)]


In [7]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "list"])

Package            Version
------------------ -----------
altgraph           0.17.2
appnope            0.1.4
asttokens          3.0.1
comm               0.2.3
debugpy            1.8.20
decorator          5.2.1
exceptiongroup     1.3.1
executing          2.2.1
future             0.18.2
importlib_metadata 8.7.1
ipykernel          6.31.0
ipython            8.18.1
jedi               0.19.2
jupyter_client     8.6.3
jupyter_core       5.8.1
macholib           1.15.2
matplotlib-inline  0.2.1
nest-asyncio       1.6.0
numpy              2.0.2
packaging          26.0
pandas             2.3.3
parso              0.8.6
pexpect            4.9.0
pip                21.2.4
platformdirs       4.4.0
prompt_toolkit     3.0.52
psutil             7.2.2
psycopg2-binary    2.9.11
ptyprocess         0.7.0
pure_eval          0.2.3
Pygments           2.20.0
python-dateutil    2.9.0.post0
pytz               2025.2
PyYAML             6.0.3
pyzmq              27.1.0
setuptools         58.0.4
six                1.17

You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


CompletedProcess(args=['/Library/Developer/CommandLineTools/usr/bin/python3', '-m', 'pip', 'list'], returncode=0)

In [1]:
import sys
print(sys.executable)

import xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path
import calendar

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

/Library/Developer/CommandLineTools/usr/bin/python3


## 2. Caminhos dos arquivos

Ajuste esta célula conforme a estrutura do seu projeto.

Se o notebook estiver em `monograph/notebooks` e a pasta `data` estiver em `monograph/data`, então `Path.cwd().parent` costuma ser suficiente.

In [4]:
from pathlib import Path

base_dir = Path("/Users/cristiane/git/mba-ai-usp/monograph")

arquivo_xml = base_dir / "data" / "raw" / "ptt.xml"
saida_dir = base_dir / "data" / "processed"

arquivo_serie_csv = saida_dir / "serie_temporal_filtrada_gbps.csv"
arquivo_resumo_csv = saida_dir / "resumo_anual_filtrado_gbps.csv"
arquivo_weekday_csv = saida_dir / "resumo_weekday_gbps.csv"
arquivo_weekend_csv = saida_dir / "comparacao_weekend_gbps.csv"

saida_dir.mkdir(parents=True, exist_ok=True)

print("Base dir:", base_dir)
print("XML:", arquivo_xml)
print("Existe?", arquivo_xml.exists())
print("Saída:", saida_dir)

Base dir: /Users/cristiane/git/mba-ai-usp/monograph
XML: /Users/cristiane/git/mba-ai-usp/monograph/data/raw/ptt.xml
Existe? True
Saída: /Users/cristiane/git/mba-ai-usp/monograph/data/processed


## 3. Leitura do arquivo XML

In [5]:
tree = ET.parse(arquivo_xml)
root = tree.getroot()

step = int(root.find("step").text)
lastupdate = int(root.find("lastupdate").text)
rows = root.find("rra/database").findall("row")

print("step (segundos):", step)
print("lastupdate:", lastupdate)
print("número de linhas:", len(rows))

step (segundos): 300
lastupdate: 1768334849
número de linhas: 2100154


## 4. Funções auxiliares

In [6]:
def parse_valor(texto):
    if texto is None or texto == "NaN":
        return None
    return float(texto)

def medicoes_esperadas_no_ano(ano, passo_segundos=300):
    dias = 366 if calendar.isleap(ano) else 365
    return (24 * 60 * 60 // passo_segundos) * dias

## 5. Reconstrução da série temporal

Cada linha do XML possui dois valores (`v`), correspondentes aos fluxos de entrada e saída.

Neste notebook, a variável principal de análise será:

`total_bps = entrada + saída`

In [8]:
n = len(rows)
start_time = lastupdate - (n * step)

serie = []

for i, row in enumerate(rows):
    timestamp = start_time + i * step
    datahora = datetime.fromtimestamp(timestamp)

    vals = row.findall("v")

    v0 = parse_valor(vals[0].text) if len(vals) > 0 else None
    v1 = parse_valor(vals[1].text) if len(vals) > 1 else None

    total = None if (v0 is None or v1 is None) else (v0 + v1)

    serie.append((datahora, total))

df = pd.DataFrame(serie, columns=["datahora", "total_bps"])
df["datahora"] = pd.to_datetime(df["datahora"])
df["year"] = df["datahora"].dt.year

df.head()

,datahora,total_bps,year
0,2006-01-26 13:17:29,NaN,2006
1,2006-01-26 13:22:29,NaN,2006
2,2006-01-26 13:27:29,NaN,2006
3,2006-01-26 13:32:29,NaN,2006
4,2006-01-26 13:37:29,NaN,2006


## 6. Verificação inicial

In [9]:
print(df.head())
print()
print(df.info())
print()
print("Período coberto:")
print(df["datahora"].min(), "até", df["datahora"].max())
print()
print("NaN em total_bps:", df["total_bps"].isna().sum())

             datahora  total_bps  year
0 2006-01-26 13:17:29        NaN  2006
1 2006-01-26 13:22:29        NaN  2006
2 2006-01-26 13:27:29        NaN  2006
3 2006-01-26 13:32:29        NaN  2006
4 2006-01-26 13:37:29        NaN  2006

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2100154 entries, 0 to 2100153
Data columns (total 3 columns):
 #   Column     Dtype         
---  ------     -----         
 0   datahora   datetime64[ns]
 1   total_bps  float64       
 2   year       int32         
dtypes: datetime64[ns](1), float64(1), int32(1)
memory usage: 40.1 MB
None

Período coberto:
2006-01-26 13:17:29 até 2026-01-13 17:02:29

NaN em total_bps: 217539


## 7. Resumo de completude anual

A completude anual é calculada com base em `total_bps`, comparando o número esperado de medições com o número de valores ausentes.

In [10]:
resumo_completude = df.groupby("year").agg(
    total_linhas=("total_bps", "size"),
    n_na_total=("total_bps", lambda x: x.isna().sum())
)

resumo_completude["esperado"] = resumo_completude.index.map(medicoes_esperadas_no_ano)
resumo_completude["completude"] = 1 - (
    resumo_completude["n_na_total"] / resumo_completude["esperado"]
)

resumo_completude["completude"] = resumo_completude["completude"].round(3)

resumo_completude

,total_linhas,n_na_total,esperado,completude
year,,,,
2006,97761,97761,105120,0.070
2007,105120,105120,105120,0.000
2008,105408,2710,105408,0.974
2009,105120,0,105120,1.000
2010,105120,0,105120,1.000
2011,105120,378,105120,0.996
2012,105408,171,105408,0.998
2013,105120,29,105120,1.000
2014,105120,107,105120,0.999


### Interpretação

A filtragem por completude é importante para evitar que anos muito incompletos distorçam as análises de tendência e os resumos anuais.

## 8. Filtragem por completude

In [11]:
anos_validos = resumo_completude.loc[resumo_completude["completude"] > 0.95].index

df_filtrado = df[df["year"].isin(anos_validos)].copy()

print("Anos válidos:", list(anos_validos))
print("Dimensão da base filtrada:", df_filtrado.shape)

Anos válidos: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
Dimensão da base filtrada: (1897273, 3)


## 9. Conversão para Gbps

A partir deste ponto, todas as análises são feitas sobre os dados filtrados e expressas em **Gbps**.

In [12]:
df_filtrado["total_gbps"] = df_filtrado["total_bps"] / 1e9
df_filtrado.head()

,datahora,total_bps,year,total_gbps
202881,2008-01-01 00:02:29,NaN,2008,NaN
202882,2008-01-01 00:07:29,NaN,2008,NaN
202883,2008-01-01 00:12:29,NaN,2008,NaN
202884,2008-01-01 00:17:29,NaN,2008,NaN
202885,2008-01-01 00:22:29,NaN,2008,NaN


## 10. Resumo anual apenas dos dados filtrados

O resumo abaixo é calculado **somente após o filtro de completude** e usa exclusivamente o tráfego total.

In [13]:
resumo = df_filtrado.groupby("year").agg(
    total_linhas=("total_gbps", "size"),
    media_total_gbps=("total_gbps", "mean"),
    mediana_total_gbps=("total_gbps", "median"),
    min_total_gbps=("total_gbps", "min"),
    max_total_gbps=("total_gbps", "max")
)

cols_gbps = ["media_total_gbps", "mediana_total_gbps", "min_total_gbps", "max_total_gbps"]
resumo[cols_gbps] = resumo[cols_gbps].round(3)

resumo

,total_linhas,media_total_gbps,mediana_total_gbps,min_total_gbps,max_total_gbps
year,,,,,
2008,105408,0.631,0.549,0.000,5.579
2009,105120,1.299,1.145,0.000,8.949
2010,105120,3.405,3.105,0.000,16.776
2011,105120,6.586,6.884,0.624,18.824
2012,105408,13.250,14.047,0.092,1001.530
2013,105120,26.415,27.944,2.903,950.902
2014,105120,65.944,66.895,5.972,783.080
2015,105120,116.210,122.704,15.791,480.673
2016,105408,214.643,232.270,34.100,1006.100


## 11. Salvando os CSVs principais

In [15]:
df_filtrado.to_csv(arquivo_serie_csv, index=False)
resumo.to_csv(arquivo_resumo_csv)

print("Série filtrada salva em:", arquivo_serie_csv)
print("Resumo anual filtrado salvo em:", arquivo_resumo_csv)


Série filtrada salva em: /Users/cristiane/git/mba-ai-usp/monograph/data/processed/serie_temporal_filtrada_gbps.csv
Resumo anual filtrado salvo em: /Users/cristiane/git/mba-ai-usp/monograph/data/processed/resumo_anual_filtrado_gbps.csv


## Arquivos gerados

Este notebook gera os conjuntos de dados processados que servirão de entrada para os notebooks de análise.

Arquivos produzidos:

- `serie_temporal_filtrada_gbps.csv`: série temporal do tráfego total, em Gbps, após aplicação do filtro de completude anual;
- `resumo_anual_filtrado_gbps.csv`: estatísticas anuais descritivas da série filtrada.

Esses arquivos são utilizados como ponto de partida pelos notebooks **01_exploratory_analysis_traffic_level** e **02_peak_analysis**, evitando a necessidade de reconstruir a série temporal a partir do arquivo XML bruto.

```mermaid
flowchart TD
    A[ptt.xml] --> B[00_data_preparation]
    B --> C[serie_temporal_filtrada_gbps.csv]
    B --> D[resumo_anual_filtrado_gbps.csv]
    C --> E[01_exploratory_analysis_traffic_level]
    C --> F[02_peak_analysis]
```